# MathForge basic SLM-versus-frontier evaluation

Runs 12 frozen technique-interaction briefs twice per model, saves all artifacts to Drive, and creates a blinded human-review packet. The frontier cell makes **24 paid calls**.

## 1. Install dependencies

In [ ]:
!pip install -q "transformers==5.13.0" "peft==0.19.1" "accelerate==1.14.0" "bitsandbytes==0.49.2" "sqlmodel>=0.0.39" "openai>=2.44.0" pylatexenc

## 2. Upload the evaluation bundle
Upload `mathforge_behavior_eval_bundle.zip` from the project root.

In [ ]:
from google.colab import files
uploaded = files.upload()
assert 'mathforge_behavior_eval_bundle.zip' in uploaded
!rm -rf /content/mathforge_behavior_eval
!mkdir -p /content/mathforge_behavior_eval
!unzip -q mathforge_behavior_eval_bundle.zip -d /content/mathforge_behavior_eval
BUNDLE = '/content/mathforge_behavior_eval'
!PYTHONPATH={BUNDLE}/src python {BUNDLE}/scripts/run_behavior_eval.py --help

## 3. Mount Drive and select the frozen adapter

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ADAPTER = '/content/drive/MyDrive/mathforge-qlora-7b-full2956-20260712-v1'  # edit if needed
EVAL_ROOT = '/content/drive/MyDrive/mathforge-evals/basic-v1'
SLM_OUTPUT = EVAL_ROOT + '/slm.jsonl'
FRONTIER_OUTPUT = EVAL_ROOT + '/frontier.jsonl'
BLIND_DIR = EVAL_ROOT + '/blind'

## 4. Generate the SLM arm

In [ ]:
!PYTHONPATH={BUNDLE}/src python {BUNDLE}/scripts/run_behavior_eval.py slm --suite {BUNDLE}/eval/behavior_suite_basic_v1.jsonl --adapter "{ADAPTER}" --output "{SLM_OUTPUT}"

## 5. Configure the frontier endpoint
Use one exact frozen model ID. If you use an Anthropic-compatible gateway, provide its base URL and bearer token; otherwise provide an Anthropic API key.

In [ ]:
import os
from getpass import getpass
FRONTIER_PROVIDER = 'anthropic'
FRONTIER_MODEL = 'claude-opus-4-8'  # replace with the exact ID exposed by your endpoint
ANTHROPIC_BASE_URL = input('Anthropic-compatible base URL (blank for api.anthropic.com): ').strip()
if ANTHROPIC_BASE_URL:
    os.environ['ANTHROPIC_BASE_URL'] = ANTHROPIC_BASE_URL
    os.environ['ANTHROPIC_AUTH_TOKEN'] = getpass('Gateway bearer token: ')
else:
    os.environ['ANTHROPIC_API_KEY'] = getpass('Anthropic API key: ')

## 6. Generate the frontier arm (24 paid calls)

In [ ]:
!PYTHONPATH={BUNDLE}/src python {BUNDLE}/scripts/run_behavior_eval.py frontier --suite {BUNDLE}/eval/behavior_suite_basic_v1.jsonl --provider {FRONTIER_PROVIDER} --model "{FRONTIER_MODEL}" --output "{FRONTIER_OUTPUT}"

## 7. Create and download the blinded review packet

In [ ]:
!PYTHONPATH={BUNDLE}/src python {BUNDLE}/scripts/run_behavior_eval.py blind --slm "{SLM_OUTPUT}" --frontier "{FRONTIER_OUTPUT}" --output-dir "{BLIND_DIR}"
files.download(BLIND_DIR + '/review_packet.md')
files.download(BLIND_DIR + '/human_labels.csv')

## 8. After blind review, upload the completed labels and report
Do not open `blind_key.json` until labeling is finished.

In [ ]:
import shutil
reviewed = files.upload()
assert 'human_labels.csv' in reviewed
shutil.copy('human_labels.csv', BLIND_DIR + '/human_labels.csv')
!PYTHONPATH={BUNDLE}/src python {BUNDLE}/scripts/run_behavior_eval.py report --blind-dir "{BLIND_DIR}"
files.download(BLIND_DIR + '/report.md')
files.download(BLIND_DIR + '/summary.json')